# 🧠 MANATUABON PHASE 9: KAGGLE LORA FINE-TUNING (Unsloth Edition)

## NVIDIA Nemotron-3-Nano-30B Reasoning Optimization
Run this notebook on **Google Colab Pro** with an **A100 GPU (80GB)**.

Upload `synthetic_cot.jsonl` (generated by `kaggle_pipeline.py`) before running.

This notebook uses **Unsloth** which natively handles the hybrid Mamba-2 + MoE architecture.

In [ ]:
# 1. Install Unsloth + Dependencies
!pip install unsloth

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.5/55.5 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.8/62.8 MB 41.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 60.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 183.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 55.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 403.2/403.2 kB 51.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 179.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 127.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.7/183.7 kB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 61.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 91.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22

In [ ]:
# 2. Load Model & Tokenizer via Unsloth
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Nemotron-3-Nano-30B-A3B",
    max_seq_length=2048,
    dtype=None,         # Auto-detect (bf16 on A100)
    load_in_4bit=True,  # Unsloth handles 4-bit cleanly for hybrid models
)

print(f"\n✅ Model loaded successfully! Tokenizer vocab size: {len(tokenizer)}")

==((====))==  Unsloth 2026.3.16: Fast Nemotron_H patching. Transformers: 5.3.0.
   \\   /|    NVIDIA RTX PRO 6000 Blackwell Server Edition. Num GPUs = 1. Max memory: 94.971 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


ValueError: Some modules are dispatched on the CPU or the disk. Make sure you have enough GPU RAM to fit the quantized model. If you want to dispatch the model on the CPU or the disk while keeping these modules in 32-bit, you need to set `llm_int8_enable_fp32_cpu_offload=True` and pass a custom `device_map` to `from_pretrained`. Check https://huggingface.co/docs/transformers/main/en/main_classes/quantization#offload-between-cpu-and-gpu for more details. 

In [ ]:
# 3. Apply LoRA Adapter (Rank 32 max per Kaggle rules)
model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    lora_alpha=64,
    lora_dropout=0,  # Unsloth optimized: dropout=0 is fastest
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_gradient_checkpointing=False, # Disable gradient checkpointing as it's not supported by this model
    random_state=42,
)
model.print_trainable_parameters()

In [ ]:
# 4. Load the CoT Dataset
import os
from datasets import load_dataset

dataset_path = "master_ensemble_dataset.jsonl"
if not os.path.exists(dataset_path):
    print("❌ ERROR: Please upload master_ensemble_dataset.jsonl to the Colab workspace!")
else:
    dataset = load_dataset("json", data_files=dataset_path, split="train")

    def format_prompt(sample):
        # Mapping the actual columns: prompt, cot, answer
        prompt = sample["prompt"]
        cot = sample["cot"]
        answer = sample["answer"]
        # Constructing the text for SFT training
        text = f"User: {prompt}\n\nAssistant: <thought>\n{cot}\n</thought>\n{answer}{tokenizer.eos_token}"
        return {"text": text}

    dataset = dataset.map(format_prompt)

    # 90/10 Train/Eval split for Early Stopping
    dataset_split = dataset.train_test_split(test_size=0.1, seed=42)
    train_data = dataset_split["train"]
    eval_data = dataset_split["test"]
    print(f"\n✅ Loaded {len(train_data)} train + {len(eval_data)} eval traces.")

In [ ]:
# 5. Configure Trainer with Early Stopping
from trl import SFTTrainer, SFTConfig
from transformers import EarlyStoppingCallback

output_dir = "./nemotron_reasoning_lora"

# Ensure variables exist before initializing trainer
if 'train_data' in globals() and 'eval_data' in globals():
    trainer = SFTTrainer(
        model=model,
        processing_class=tokenizer,
        train_dataset=train_data,
        eval_dataset=eval_data,
        args=SFTConfig(
            output_dir=output_dir,
            dataset_text_field="text",
            per_device_train_batch_size=1,
            gradient_accumulation_steps=4,
            warmup_steps=10,
            max_steps=150,
            learning_rate=2e-4,
            bf16=True,
            logging_steps=5,
            eval_strategy="steps",
            eval_steps=10,
            save_strategy="steps",
            save_steps=10,
            load_best_model_at_end=True,
            metric_for_best_model="eval_loss",
            optim="adamw_8bit",
            seed=42,
            gradient_checkpointing=False,
        ),
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
    )

    # 6. Train!
    print("\n🔥 Starting LoRA Fine-tuning with Early Stopping...")
    trainer_stats = trainer.train()
    print(f"\n✅ Training complete! Total steps: {trainer_stats.global_step}")
else:
    print("❌ ERROR: train_data and eval_data not found. Please run the data loading cell first.")

In [ ]:
# 7. Save the LoRA Adapter for Kaggle Submission
print(f"Saving LoRA adapter to {output_dir}...")
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print("\n✅ LoRA adapter saved! Ready to zip and submit to Kaggle! 🏆")

# Mount google drive
from google.colab import drive
drive.mount('/content/drive')

#SAVE MODEL IMMEDIATELY AFTER TRAINING
# Save to Drive
import shutil
shutil.copytree('nemotron_reasoning_lora', '/content/drive/MyDrive/nemotron_reasoning_lora', dirs_exist_ok=True)
print('Saved to Drive ✅'),

Saving LoRA adapter to ./nemotron_reasoning_lora...

✅ LoRA adapter saved! Ready to zip and submit to Kaggle! 🏆
Mounted at /content/drive
Saved to Drive ✅


(None,)

In [ ]:
# 8. (Optional) Quick Inference Test
FastLanguageModel.for_inference(model)

test_prompt = "Find the next number in the sequence: 2, 6, 12, 20, 30, ?"
inputs = tokenizer(f"User: {test_prompt}\n\nAssistant:", return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=512, temperature=0.1)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Both `max_new_tokens` (=512) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


User: Find the next number in the sequence: 2, 6, 12, 20, 30, ?

Assistant: <thought>
<think>
Let me analyze the given sequence: 2, 6, 12, 20, 30, ?

First, I'll look at the differences between consecutive terms:
- 6 - 2 = 4
- 12 - 6 = 6
- 20 - 12 = 8
- 30 - 20 = 10

The differences are: 4, 6, 8, 10
This appears to be increasing by 2 each time.

If this pattern continues, the next difference should be 12 (since 10 + 2 = 12).

So, the next term would be: 30 + 12 = 42

Let me verify this by looking for other patterns.

Another approach: Let's see if there's a formula involving multiplication or other operations.

Looking at the numbers:
- 2 = 1×2
- 6 = 2×3
- 12 = 3×4
- 20 = 4×5
- 30 = 5×6

This confirms the pattern: each term is n×(n+1) where n starts from 1.

For the 6th term, n would be 6:
6×7 = 42

Both approaches lead to the same answer: 42.
</think>
\boxed{42}
</thought>
42


In [ ]:
import os
import shutil

adapter_dir = "nemotron_reasoning_lora"

# 1. Delete ALL files except the two vital adapter files
allowed_files = ["adapter_config.json", "adapter_model.safetensors"]

for item in os.listdir(adapter_dir):
    item_path = os.path.join(adapter_dir, item)
    if os.path.isfile(item_path) and item not in allowed_files:
        os.remove(item_path)
        print(f"🗑️ Deleted conflicting file: {item}")
    elif os.path.isdir(item_path):
        shutil.rmtree(item_path)
        print(f"🗑️ Deleted trailing folder: {item}")

# 2. Check the directory to ensure only 2 files remain
print("\nFiles ready for zipping:", os.listdir(adapter_dir))

# 3. Zip the clean files
shutil.make_archive("submission_clean", "zip", adapter_dir)
shutil.copy("submission_clean.zip", "/content/drive/MyDrive/submission_clean.zip")

print("\n✅ Clean submission_clean.zip created! Upload this one to Kaggle! 🏆")



Files ready for zipping: ['adapter_model.safetensors', 'adapter_config.json']

✅ Clean submission_clean.zip created! Upload this one to Kaggle! 🏆
